## Summary of outputs
- `Total rows` : CSV 的總筆數。
- `Number of session_id values that repeat` : 若大於 0，代表有重複的 session_id，下面會列出部分例子。
- `Relationship counts` : 每個 relationship 樣態的計數，並顯示 `familiar` 與 `stranger` 的明確數字。

In [18]:
# Section 1: Import Required Libraries and dataloader.py
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import json
import librosa
import cv2
from pathlib import Path

# Ensure dataloader.py and config are accessible
sys.path.append(os.path.abspath('../scripts'))
sys.path.append(os.path.abspath('..'))
from dataloader import InteractionDataLoader
from config import SESSION_CONFIG

In [19]:
# Section 3: List Available File IDs in Session Groups
from pathlib import Path
import glob

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
ASSET_DIR = PROJECT_ROOT / 'assets'
SESSION_GROUP_DIR = DATA_DIR / 'session_groups' / 'improvised' / 'dev' / '0000'

json_files = glob.glob(str(SESSION_GROUP_DIR / '*/' / '*.json'))
file_ids = [os.path.splitext(os.path.basename(f))[0] for f in json_files]
print(f"Found {len(file_ids)} file IDs:")
print(file_ids)

Found 10 file IDs:
['V00_S0809_I00000126_P0947', 'V00_S0700_I00000576_P0844A', 'V01_S1881_I00000186_P2767', 'V00_S0925_I00000488_P0816', 'V00_S0809_I00000785_P0383', 'V01_S0935_I00001226_P2040', 'V00_S1061_I00000160_P0383', 'V00_S0648_I00000382_P0801', 'V00_S2020_I00000679_P1276A', 'V00_S0696_I00000543_P0844A']


# Looking around randomly

In [27]:
import pandas as pd

relationship_df = pd.read_csv(ASSET_DIR / 'relationships.csv')

# 1) total rows
total_rows = len(relationship_df)
print(f"Total rows: {total_rows}")

# 2) check duplicate session_id
session_dup_counts = relationship_df['session_id'].value_counts()
duplicate_session_ids = session_dup_counts[session_dup_counts > 1]
num_duplicate_session_ids = duplicate_session_ids.shape[0]
print(f"Number of session_id values that repeat: {num_duplicate_session_ids}")

# 3) relationship counts (overall and specifically familiar/stranger)
relationship_counts = relationship_df['relationship'].value_counts()
familiar_count = relationship_counts.get('familiar', 0)
stranger_count = relationship_counts.get('stranger', 0)


if num_duplicate_session_ids > 0:
    print("Example duplicated session_id and counts:")
    print(duplicate_session_ids.head(20).to_string())
    # additionally print all rows for the first duplicated session_id for inspection
    first_dup = duplicate_session_ids.index[0]
    print(f"\nRows for example duplicated session_id: {first_dup}")
    print(df[df['session_id']==first_dup].to_string(index=False))
print('\nRelationship counts:')
print(relationship_counts.to_string())
print(f"\n'familiar' count: {familiar_count}")
print(f"'stranger' count: {stranger_count}")

Total rows: 5098
Number of session_id values that repeat: 1777
Example duplicated session_id and counts:
session_id
477    3
271    3
144    3
211    3
263    3
213    3
209    3
262    3
282    3
258    3
217    3
176    3
212    3
261    3
270    3
276    3
259    3
210    3
281    3
264    3

Rows for example duplicated session_id: 477
vendor_id  session_id relationship relationship_detail
      V03         477     stranger            stranger
      V00         477     familiar      family-generic
      V01         477     stranger            stranger

Relationship counts:
relationship
stranger    3347
familiar    1751

'familiar' count: 1751
'stranger' count: 3347


In [28]:
s=relationship_df['session_id'].value_counts()
dup_ids = s[s>1].index.tolist()
print('NUM_DUPLICATE_SESSION_IDS:', len(dup_ids))
if len(dup_ids)==0:
    print('No duplicated session_id found')
else:
    sid = dup_ids[3]
    print('Example duplicated session_id:', sid, 'count:', s[sid])
    # print all rows for that session_id
    rows = relationship_df[relationship_df['session_id']==sid]
    print('\nRows for session_id', sid)
    print(rows.to_string(index=False))

NUM_DUPLICATE_SESSION_IDS: 1777
Example duplicated session_id: 211 count: 3

Rows for session_id 211
vendor_id  session_id relationship relationship_detail
      V03         211     stranger            stranger
      V00         211     familiar             friends
      V01         211     stranger            stranger


重複：同組人但關係卻不同？（補充：同個人的大五也可能不同）

# Check every csv

In [190]:
filelist_df = pd.read_csv(ASSET_DIR / 'filelist.csv')
print(len(filelist_df))

# Extract individual ID components from file_id
# Format: V{vendor_id}_S{session_id}_I{interaction_id}_P{participant_id}
import re

def extract_ids(file_id):
    """Extract vendor_id, session_id, interaction_id (prompt hash), participant_id from file_id string"""
    pattern = r'V(\d+)_S(\d+)_I(\d+)_P(\w+)'
    match = re.match(pattern, file_id)
    if match:
        return match.groups()
    else:
        return None, None, None, None

# Apply extraction to all file_ids
id_components = filelist_df['file_id'].apply(extract_ids)
filelist_df[['vendor_id_raw', 'session_id', 'interaction_id', 'participant_id']] = pd.DataFrame(id_components.tolist(), index=filelist_df.index)

# Create two versions of vendor_id
# vendor_id: numeric format (00, 01, 02, 03)
filelist_df['vendor_id'] = filelist_df['vendor_id_raw'].astype(int)

# vendor_Vid: V + numeric format (V00, V01, V02, V03)  
filelist_df['vendor_Vid'] = 'V' + filelist_df['vendor_id_raw']

# Convert numeric IDs to appropriate data types (except vendor_id which should remain as zero-padded string)
filelist_df['session_id'] = pd.to_numeric(filelist_df['session_id'], errors='coerce')
filelist_df['interaction_id'] = pd.to_numeric(filelist_df['interaction_id'], errors='coerce')
# participant_id stays as string since it can have letters (e.g., P0844A)

# Drop the temporary raw vendor_id column
filelist_df = filelist_df.drop('vendor_id_raw', axis=1)

print(f"Successfully extracted ID components for {len(filelist_df)} rows")
print(f"Columns now include: {list(filelist_df.columns)}")
print(f"vendor_id examples: {filelist_df['vendor_id'].unique()[:10]}")
print(f"vendor_Vid examples: {filelist_df['vendor_Vid'].unique()[:10]}")

filelist_df.head()

129572
Successfully extracted ID components for 129572 rows
Columns now include: ['file_id', 'label', 'split', 'batch_idx', 'archive_idx', 'has_imitator_movement', 'has_annotation_1p', 'has_annotation_3p', 'session_id', 'interaction_id', 'participant_id', 'vendor_id', 'vendor_Vid']
vendor_id examples: [0 1 3 2]
vendor_Vid examples: ['V00' 'V01' 'V03' 'V02']


,file_id,label,split,batch_idx,archive_idx,has_imitator_movement,has_annotation_1p,has_annotation_3p,session_id,interaction_id,participant_id,vendor_id,vendor_Vid
0,V00_S0644_I00000129_P0799,improvised,dev,0,0,1,0,0,644,129,0799,0,V00
1,V00_S0692_I00000536_P0844A,improvised,dev,0,0,1,0,0,692,536,0844A,0,V00
2,V00_S0809_I00000309_P0947,improvised,dev,0,0,1,0,0,809,309,0947,0,V00
3,V00_S0925_I00000479_P0816,improvised,dev,0,0,1,0,0,925,479,0816,0,V00
4,V00_S0925_I00000538_P0383,improvised,dev,0,0,1,0,0,925,538,0383,0,V00


In [191]:
interactions_df = pd.read_csv(ASSET_DIR / 'interactions.csv')
print(len(interactions_df))
interactions_df.head()

1312


,Unnamed: 0,prompt_hash,prompt_id_unique,participant_a_prompt_text,participant_b_prompt_text,ipc_a,ipc_b,interaction_type
0,0,0,P0070_v3.4.fam_ANCP_XXXX,"Tell your partner about a time when you went out of your way to make someone feel included or welcome in a social situation. What did you do, and how did the person respond? What did you learn from the experience?","Your partner will tell you about a time they made an effort to include someone. Discuss with them the challenges and rewards of being hospitable in social situations, and explore ways to create a more inclusive environment for others.",ANCP,NaN,ipc_conversation
1,1,1,P0066_v3.4.fam_AMCM_XXXX,Your partner will tell you about a time they felt like they didn't speak up or assert themselves. Discuss with them what they could have done differently and how you can support each other in similar situations.,Recall a conversation or situation where you felt like you didn't speak up or assert yourself as much as you should have. Describe what happened and what you would do differently if faced with a similar situation in the future.,AMCM,NaN,ipc_conversation
2,2,3,P0065_v3.4.fam_APCN_XXXX,Your partner will tell you about a cause they're passionate about. Listen to their perspective and discuss potential strategies for making progress on the issue.,"Discuss a topic or issue that you feel strongly about and have been advocating for over time. What is it about this issue that motivates you to keep fighting for it, and what progress have you seen?",APCN,NaN,ipc_conversation
3,3,4,P0002_v3.4.aac_ANCP_XXXX-148,"You're going to play a game with your partner. This game is going to test your acting skills!\n\nYou are each given different lists of sentences below.\n\nNotice that one of the words is bolded in each sentence. When it is your turn you will read one of the sentences out loud AND act-out a corresponding gesture of the bolded word, emphasizing it. Your partner will react by responding with a word or phrase out-loud and making a corresponding expression or gesture.\n\nExample:\nYou: (Sentence is “I’m feeling *confused*.”) You say it out loud while frowning and raising your eyebrows at the word “confused.”\nPartner: Responds saying “Why?” and acts with a concerned facial expression.\n\nAlternate with your partner until you’ve completed your list or the moderator asks you to move on.\n\nYou will go first.\n\nHere's your list:\nThe *bulky* sweater kept him warm during the winter.\nHe is typing on the *computer*.\nShe is *rinsing* the vegetables.\nShe is *walking* to the store.\nShe is filled with *love*.\nHe is fidgeting with *anxiety*.\nThe music is *mellow*.\nShe is visibly *stressed*.\nHe is *farewelling* his friends.\nShe had *a copious amount of* notes for the exam.","You're going to play a game with your partner. This game is going to test your acting skills!\n\nYou are each given different lists of sentences below.\n\nNotice that one of the words is bolded in each sentence. When it is your turn you will read one of the sentences out loud AND act-out a corresponding gesture of the bolded word, emphasizing it. Your partner will react by responding with a word or phrase out-loud and making a corresponding expression or gesture.\n\nExample:\nYou: (Sentence is “I’m feeling *confused*.”) You say it out loud while frowning and raising your eyebrows at the word “confused.”\nPartner: Responds saying “Why?” and acts with a concerned facial expression.\n\nAlternate with your partner until you’ve completed your list or the moderator asks you to move on.\n\nYour partner will go first.\n\nHere's your list:\nShe is *accepting* the offer.\nShe carried a *compact* umbrella that fit easily in her bag.\nShe is driving the *car*.\nHe is *peeling* the potatoes.\nShe is gasping in *surprise*.\nHe is showing *disgust*.\nHe is speaking with *enthusiasm*.\nThe music is *gentle*.\nHe is *forgiving* the mistake.\nThey are *gossiping* quietly.",ANCP,NaN,grounded_gesture
4,4,5,P0074_v3.4.fam_ANCM_XX

In [192]:
participant_df = pd.read_csv(ASSET_DIR / 'participants.csv')
print(len(participant_df))
participant_df.head()

4284


,extraversion_raw,agreeableness_raw,conscientiousness_raw,neuroticism_raw,openness_raw,vendor_id,participant_id
0,4.666666666666667,4.416666666666667,5.0,1.3333333333333333,4.5,0,0300
1,Undisclosed,Undisclosed,Undisclosed,Undisclosed,Undisclosed,3,3212
2,Undisclosed,Undisclosed,Undisclosed,Undisclosed,Undisclosed,0,0844A
3,Undisclosed,Undisclosed,Undisclosed,Undisclosed,Undisclosed,3,2814
4,Undisclosed,Undisclosed,Undisclosed,Undisclosed,Undisclosed,3,4948


In [193]:
relationship_df = pd.read_csv(ASSET_DIR / 'relationships.csv')
print(len(relationship_df))
relationship_df.head()

5098


,vendor_id,session_id,relationship,relationship_detail
0,V03,1805,familiar,coworkers
1,V03,1936,familiar,coworkers
2,V03,1380,familiar,familiar-generic
3,V03,1334,stranger,stranger
4,V03,1381,familiar,familiar-generic


# Merge to a single seamless_interaction.csv

In [198]:
filelist_interaction_df = filelist_df.merge(
    interactions_df,
    left_on="interaction_id",
    right_on='prompt_hash',
    how='left',
    suffixes=("", "_right")
)
filelist_interaction_df.head()

,file_id,label,split,batch_idx,archive_idx,has_imitator_movement,has_annotation_1p,has_annotation_3p,session_id,interaction_id,participant_id,vendor_id,vendor_Vid,Unnamed: 0,prompt_hash,prompt_id_unique,participant_a_prompt_text,participant_b_prompt_text,ipc_a,ipc_b,interaction_type
0,V00_S0644_I00000129_P0799,improvised,dev,0,0,1,0,0,644,129,0799,0,V00,122,129,P0063_AA_ANCP_XXXX,"Ask your partner what they truly want in life? What are their deepest, most profound dreams? Try to\nfind similarities or common ground with your own dreams.",Your partner will ask you a question that requires some thought. Take your time and discuss your\nanswer with them.,ANCP,NaN,charades
1,V00_S0692_I00000536_P0844A,improvised,dev,0,0,1,0,0,692,536,0844A,0,V00,536,536,P0048_CC_ANCP_APCN,"You will each see the same prompt:\n\nDiscuss with your partner - ""if we were to start a company or business together what would it be? How would we make it succeed?""","You will each see the same prompt:\n\nDiscuss with your partner - ""if we were to start a company or business together what would it be? How would we make it succeed?""\n\nAct as if you are an authority on the subject. Call yourself an entrepreneur and try to lead the conversation.",ANCP,APCN,ipc_conversation
2,V00_S0809_I00000309_P0947,improvised,dev,0,0,1,0,0,809,309,0947,0,V00,302,309,P0049_CC_ANCP_XXXX,"You will each see the same prompt:\n\nThink about something in the world you and your partner both care about. Could be a ""big"" thing like climate change or a ""small thing"" like filling potholes on your local road. Bring up the topic and discuss how you might improve it together.","You will each see the same prompt:\n\nThink about something in the world you and your partner both care about. Could be a ""big"" thing like climate change or a ""small thing"" like filling potholes on your local road. Bring up the topic and discuss how you might improve it together.",ANCP,NaN,ipc_conversation
3,V00_S0925_I00000479_P0816,improvised,dev,0,0,1,0,0,925,479,0816,0,V00,478,479,P0058_DN_ANCM_XXXX,Discuss your morning routine with your partner. Then ask them about their morning routine. Find at least one element that differs and discuss what this says about your\r\npersonalities.,Your partner will tell you about their morning routine and then ask you about yours. Discuss together to identify at least one element that differs and discuss what this\r\nsays about your personalities.,ANCM,NaN,ipc_conversation
4,V00_S0925_I00000538_P0383,improvised,dev,0,0,1,0,0,925,538,0383,0,V00,538,538,P0016_MP_AMCN_APCP,"Recall a time that you were stuck in life or on a project and someone helped you to get through it. Tell the story about this time, describe how the person helped you, and what you learned from the experience.","Your partner is going to tell a story of overcoming a challenge with someone's help. Ignoring that someone, insist it was your partner's own qualities that got them through. Really build them up in response. Shower them in praise.",AMCN,APCP,ipc_conversation


In [199]:
filelist_interaction_participant_df = filelist_interaction_df.merge(
    participant_df,
    left_on=["vendor_id","participant_id"],
    right_on=['vendor_id','participant_id'],
    how='left',
    suffixes=("", "_right")
)
filelist_interaction_participant_df.head()

,file_id,label,split,batch_idx,archive_idx,has_imitator_movement,has_annotation_1p,has_annotation_3p,session_id,interaction_id,participant_id,vendor_id,vendor_Vid,Unnamed: 0,prompt_hash,prompt_id_unique,participant_a_prompt_text,participant_b_prompt_text,ipc_a,ipc_b,interaction_type,extraversion_raw,agreeableness_raw,conscientiousness_raw,neuroticism_raw,openness_raw
0,V00_S0644_I00000129_P0799,improvised,dev,0,0,1,0,0,644,129,0799,0,V00,122,129,P0063_AA_ANCP_XXXX,"Ask your partner what they truly want in life? What are their deepest, most profound dreams? Try to\nfind similarities or common ground with your own dreams.",Your partner will ask you a question that requires some thought. Take your time and discuss your\nanswer with them.,ANCP,NaN,charades,3.1666666666666665,3.8333333333333335,3.0833333333333335,2.8333333333333335,3.5
1,V00_S0692_I00000536_P0844A,improvised,dev,0,0,1,0,0,692,536,0844A,0,V00,536,536,P0048_CC_ANCP_APCN,"You will each see the same prompt:\n\nDiscuss with your partner - ""if we were to start a company or business together what would it be? How would we make it succeed?""","You will each see the same prompt:\n\nDiscuss with your partner - ""if we were to start a company or business together what would it be? How would we make it succeed?""\n\nAct as if you are an authority on the subject. Call yourself an entrepreneur and try to lead the conversation.",ANCP,APCN,ipc_conversation,Undisclosed,Undisclosed,Undisclosed,Undisclosed,Undisclosed
2,V00_S0809_I00000309_P0947,improvised,dev,0,0,1,0,0,809,309,0947,0,V00,302,309,P0049_CC_ANCP_XXXX,"You will each see the same prompt:\n\nThink about something in the world you and your partner both care about. Could be a ""big"" thing like climate change or a ""small thing"" like filling potholes on your local road. Bring up the topic and discuss how you might improve it together.","You will each see the same prompt:\n\nThink about something in the world you and your partner both care about. Could be a ""big"" thing like climate change or a ""small thing"" like filling potholes on your local road. Bring up the topic and discuss how you might improve it together.",ANCP,NaN,ipc_conversation,4.25,4.25,3.8333333333333335,2.0,4.583333333333333
3,V00_S0925_I00000479_P0816,improvised,dev,0,0,1,0,0,925,479,0816,0,V00,478,479,P0058_DN_ANCM_XXXX,Discuss your morning routine with your partner. Then ask them about their morning routine. Find at least one element that differs and discuss what this says about your\r\npersonalities.,Your partner will tell you about their morning routine and then ask you about yours. Discuss together to identify at least one element that differs and discuss what this\r\nsays about your personalities.,ANCM,NaN,ipc_conversation,3.9166666666666665,3.25,4.0,2.0,3.5833333333333335
4,V00_S0925_I00000538_P0383,improvised,dev,0,0,1,0,0,925,538,0383,0,V00,538,538,P0016_MP_AMCN_APCP,"Recall a time that you were stuck in life or on a project and someone helped you to get through it. Tell the story about this time, describe how the person helped you, and what you learned from the experience.","Your partner is going to tell a story of overcoming a challenge with someone's help. Ignoring that someone, insist it was your partner's own qualities that got them through. Really build them up in response. Shower them in praise.",AMCN,APCP,ipc_conversation,4.25,4.166666666666667,3.8333333333333335,1.5833333333333333,3.5833333333333335


In [200]:
filelist_interaction_participant_relationship_df = filelist_interaction_participant_df.merge(
    relationship_df,
    left_on=["vendor_Vid","session_id"],
    right_on=['vendor_id','session_id'],
    how='left',
    suffixes=("", "_right")
)
filelist_interaction_participant_relationship_df.head()

,file_id,label,split,batch_idx,archive_idx,has_imitator_movement,has_annotation_1p,has_annotation_3p,session_id,interaction_id,participant_id,vendor_id,vendor_Vid,Unnamed: 0,prompt_hash,prompt_id_unique,participant_a_prompt_text,participant_b_prompt_text,ipc_a,ipc_b,interaction_type,extraversion_raw,agreeableness_raw,conscientiousness_raw,neuroticism_raw,openness_raw,vendor_id_right,relationship,relationship_detail
0,V00_S0644_I00000129_P0799,improvised,dev,0,0,1,0,0,644,129,0799,0,V00,122,129,P0063_AA_ANCP_XXXX,"Ask your partner what they truly want in life? What are their deepest, most profound dreams? Try to\nfind similarities or common ground with your own dreams.",Your partner will ask you a question that requires some thought. Take your time and discuss your\nanswer with them.,ANCP,NaN,charades,3.1666666666666665,3.8333333333333335,3.0833333333333335,2.8333333333333335,3.5,V00,stranger,stranger
1,V00_S0692_I00000536_P0844A,improvised,dev,0,0,1,0,0,692,536,0844A,0,V00,536,536,P0048_CC_ANCP_APCN,"You will each see the same prompt:\n\nDiscuss with your partner - ""if we were to start a company or business together what would it be? How would we make it succeed?""","You will each see the same prompt:\n\nDiscuss with your partner - ""if we were to start a company or business together what would it be? How would we make it succeed?""\n\nAct as if you are an authority on the subject. Call yourself an entrepreneur and try to lead the conversation.",ANCP,APCN,ipc_conversation,Undisclosed,Undisclosed,Undisclosed,Undisclosed,Undisclosed,V00,stranger,stranger
2,V00_S0809_I00000309_P0947,improvised,dev,0,0,1,0,0,809,309,0947,0,V00,302,309,P0049_CC_ANCP_XXXX,"You will each see the same prompt:\n\nThink about something in the world you and your partner both care about. Could be a ""big"" thing like climate change or a ""small thing"" like filling potholes on your local road. Bring up the topic and discuss how you might improve it together.","You will each see the same prompt:\n\nThink about something in the world you and your partner both care about. Could be a ""big"" thing like climate change or a ""small thing"" like filling potholes on your local road. Bring up the topic and discuss how you might improve it together.",ANCP,NaN,ipc_conversation,4.25,4.25,3.8333333333333335,2.0,4.583333333333333,V00,stranger,stranger
3,V00_S0925_I00000479_P0816,improvised,dev,0,0,1,0,0,925,479,0816,0,V00,478,479,P0058_DN_ANCM_XXXX,Discuss your morning routine with your partner. Then ask them about their morning routine. Find at least one element that differs and discuss what this says about your\r\npersonalities.,Your partner will tell you about their morning routine and then ask you about yours. Discuss together to identify at least one element that differs and discuss what this\r\nsays about your personalities.,ANCM,NaN,ipc_conversation,3.9166666666666665,3.25,4.0,2.0,3.5833333333333335,V00,stranger,stranger
4,V00_S0925_I00000538_P0383,improvised,dev,0,0,1,0,0,925,538,0383,0,V00,538,538,P0016_MP_AMCN_APCP,"Recall a time that you were stuck in life or on a project and someone helped you to get through it. Tell the story about this time, describe how the person helped you, and what you learned from the experience.","Your partner is going to tell a story of overcoming a challenge with someone's help. Ignoring that someone, insist it was your partner's own qualities that got them through. Really build them up in response. Shower them in praise.",AMCN,APCP,ipc_conversation,4.25,4.166666666666667,3.8333333333333335,1.5833333333333333,3.5833333333333335,V00,stranger,stranger


In [201]:
seamless_interaction_df = filelist_interaction_participant_relationship_df.drop(columns=["interaction_id","Unnamed: 0","vendor_id_right"])
# Save
seamless_interaction_df.to_csv(ASSET_DIR / 'seamless_interaction.csv', index=False)

In [212]:
# Basic QA for seamless_interaction_df — show only columns that have nulls and perform duplicate checks
print('Basic null-check and duplicate-check for seamless_interaction_df')
print('='*80)

try:
    print('Shape:', seamless_interaction_df.shape)
    total_rows = len(seamless_interaction_df)

    # Nulls
    null_counts = seamless_interaction_df.isnull().sum()
    null_pct = (null_counts / total_rows * 100).round(3)
    null_df = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})

    # Keep only columns with at least one null
    null_df = null_df[null_df['null_count'] > 0].sort_values('null_count', ascending=False)

    if null_df.empty:
        print('No columns contain null values.')
    else:
        print(f"Columns with nulls: {len(null_df)} (showing all)")
        print(null_df.to_string())

    print('\n' + '-'*80)
    # Duplicate checks
    print('\nDuplicate checks:')
    # 1) Exact duplicate rows (any duplicate, keep=False counts all rows that are duplicates)
    exact_dup_mask = seamless_interaction_df.duplicated(keep=False)
    exact_dup_count = int(exact_dup_mask.sum())
    print(f"Exact duplicated rows (rows that have at least one duplicate): {exact_dup_count}")

    # 2) Unique duplicate groups count (how many groups of identical rows)
    if exact_dup_count > 0:
        dup_groups = seamless_interaction_df[exact_dup_mask].groupby(list(seamless_interaction_df.columns)).size()
        print(f"Number of distinct duplicated row-groups: {len(dup_groups)}")

    # 3) Duplicates by common key sets
    key_sets = [
        ['file_id'],
        ['vendor_Vid','session_id','participant_id'],
        ['vendor_id','participant_id'],
        ['prompt_hash']
    ]
    existing_key_sets = [ks for ks in key_sets if all(k in seamless_interaction_df.columns for k in ks)]
    print('\nChecking duplicates for key sets present:')
    for ks in existing_key_sets:
        group_sizes = seamless_interaction_df.groupby(ks).size()
        dup_groups = group_sizes[group_sizes > 1].sort_values(ascending=False)
        print(f" - {ks}: groups_with_multiple_rows={len(dup_groups)} | unique_keys={group_sizes.shape[0]}")
        if len(dup_groups) > 0:
            # show top 10 duplicated groups with counts
            top10 = dup_groups.head(10)
            print('   Top duplicated groups (up to 10):')
            for idx, cnt in top10.items():
                print(f'     {idx} -> {cnt}')

    # 4) Quick check for merge-suffix leftovers
    suffix_candidates = [c for c in seamless_interaction_df.columns if c.endswith(('_right','_y','_x'))]
    print('\nColumns with merge-suffix candidates:', suffix_candidates)

    print('\nQA complete')
except NameError:
    print('seamless_interaction_df is not defined in the notebook environment. Make sure the merge cells ran and assigned it.')
except Exception as e:
    print('Error during QA:', e)

Basic null-check and duplicate-check for seamless_interaction_df
Shape: (129572, 26)
Columns with nulls: 8 (showing all)
                       null_count  null_pct
ipc_b                      103731    80.057
relationship                 3040     2.346
relationship_detail          3040     2.346
extraversion_raw             2700     2.084
agreeableness_raw            2700     2.084
conscientiousness_raw        2700     2.084
neuroticism_raw              2700     2.084
openness_raw                 2700     2.084

--------------------------------------------------------------------------------

Duplicate checks:
Exact duplicated rows (rows that have at least one duplicate): 0

Checking duplicates for key sets present:
 - ['file_id']: groups_with_multiple_rows=201 | unique_keys=129370
   Top duplicated groups (up to 10):
     V03_S0139_I00000477_P1302 -> 3
     V00_S0155_I00000461_P0223 -> 2
     V03_S0997_I00000135_P3430 -> 2
     V03_S0921_I00000321_P3165 -> 2
     V03_S0921_I00000327_P

### Note: 有些 session 的 vendor 沒有標到受試者關係

In [214]:
# Show examples where relationship or relationship_detail is null
print('Rows with missing relationship or relationship_detail')
print('='*60)

mask = seamless_interaction_df['relationship'].isnull() | seamless_interaction_df['relationship_detail'].isnull()
missing_df = seamless_interaction_df[mask]
print(f"Total rows with missing relationship fields: {len(missing_df)}")

# Show top vendor/session groups causing the missing values
if len(missing_df) > 0:
    grp = missing_df.groupby(['vendor_Vid','session_id']).size().sort_values(ascending=False)
    print('\nTop vendor_Vid/session groups with missing relationship (top 10):')
    print(grp.head(10).to_string())

    # Show example rows (select relevant columns)
    cols = [c for c in ['file_id','vendor_Vid','vendor_id','session_id','participant_id','relationship','relationship_detail'] if c in missing_df.columns]
    print('\nExample rows (up to 20):')
    print(missing_df[cols].head(20).to_string(index=False))
else:
    print('No missing rows found')

Rows with missing relationship or relationship_detail
Total rows with missing relationship fields: 3040

Top vendor_Vid/session groups with missing relationship (top 10):
vendor_Vid  session_id
V01         1264          40
            1874          40
            1553          40
            1267          40
            1607          40
            1876          40
            1255          40
            1601          38
            1541          38
            1290          38

Example rows (up to 20):
                  file_id vendor_Vid  vendor_id  session_id participant_id relationship relationship_detail
V01_S1881_I00000179_P2767        V01          1        1881           2767          NaN                 NaN
V01_S1881_I00000193_P2766        V01          1        1881           2766          NaN                 NaN
V01_S1881_I00000625_P2766        V01          1        1881           2766          NaN                 NaN
V01_S1881_I00000180_P2766        V01          1        188

# Merge annotation

In [219]:
annotation_df = pd.read_csv(ASSET_DIR / '1p_3p_json.csv')
print("annotation_df rows:", len(annotation_df))
annotation_df.head()

annotation_df rows: 293


,id,annotations:1P-IS,annotations:1P-R,annotations:3P-IS,annotations:3P-R,annotations:3P-V
0,V03_S1035_I00000375_P3095,Deep in conversation.,topic interesting to me.,Participant is engaged and explaining with attention and focus.,Participant is emphasizing their explanation while maintaining focus and awareness of surroundings.,"Participant speaks and does hand gestures, looks around, and maintains a focused, normal facial expression."
1,V03_S1039_I00000373_P3206,"Searching for words, invested",Making sure I won't stumble on words while recalling names for future examples,Participant is in doubt,Participant is trying to recall names for the future,Participant moves their arm up and down multiple times gesturing size
2,V03_S1004_I00000541_P2024,"sure, maybe opinionated",speaking about film & art,The participant is demonstrative,Because the participant wanted to accentuate the role of the quality of the story,The participant curled the fingers and raised the hands to the chest's height like they were holding something
3,V03_S1004_I00000372_P2024,"uplifted, happy, proud",to find a kindered heart,The participant is confident,The participant had a lot of knowledge in the topic and they were talking smoothly,The participant moved their hands non-stop
4,V03_S1035_I00000371_P3095,Felt like i wanted to go back to the old days.,Due to picturing the Yankees and being a former baseball player.,"Participant is calm, focused, and patiently listening.",Participant is absorbing the information with calm attention and minimal distraction.,"Participant listens attentively with minimal body movement and slight finger movement, then wipes their face with their forearm."


In [220]:
filtered_si_df = seamless_interaction_df[(seamless_interaction_df['has_annotation_1p'] == 1) & (seamless_interaction_df['has_annotation_3p'] == 1)]
print(filtered_si_df.shape)
filtered_si_df.head()

(499, 26)


,file_id,label,split,batch_idx,archive_idx,has_imitator_movement,has_annotation_1p,has_annotation_3p,session_id,participant_id,vendor_id,vendor_Vid,prompt_hash,prompt_id_unique,participant_a_prompt_text,participant_b_prompt_text,ipc_a,ipc_b,interaction_type,extraversion_raw,agreeableness_raw,conscientiousness_raw,neuroticism_raw,openness_raw,relationship,relationship_detail
38,V03_S0203_I00000537_P1437,improvised,dev,0,3,0,1,1,203,1437,3,V03,537,P0018_MN_AMCP_AMCN,"Share an embarrassing story about yourself with your partner. Without revealing personal details like names or dates, describe what happened, why it was embarrassing, and how it made you feel.",Your partner is going to tell you a story. React with complete uncomprehension. Like you don't understand why the events they are telling you made them feel the way they did.,AMCP,AMCN,ipc_conversation,Undisclosed,Undisclosed,Undisclosed,Undisclosed,Undisclosed,stranger,stranger
47,V03_S0203_I00000479_P1436,improvised,dev,0,4,0,1,1,203,1436,3,V03,479,P0058_DN_ANCM_XXXX,Discuss your morning routine with your partner. Then ask them about their morning routine. Find at least one element that differs and discuss what this says about your\r\npersonalities.,Your partner will tell you about their morning routine and then ask you about yours. Discuss together to identify at least one element that differs and discuss what this\r\nsays about your personalities.,ANCM,NaN,ipc_conversation,3.75,4.25,3.1666666666666665,3.75,3.8333333333333335,stranger,stranger
77,V03_S0203_I00000495_P1436,improvised,dev,0,7,0,1,1,203,1436,3,V03,495,P0027_MP_APCP_XXXX,Think about someone you look up to. Could be someone from your childhood or someone you look up to today. Tell a story about this person that provides rich details for your partner describing who this person is and why you look up to them.,Your partner will tell you a story about someone. Listen to them and engage with the story.,APCP,NaN,ipc_conversation,3.75,4.25,3.1666666666666665,3.75,3.8333333333333335,stranger,stranger
84,V03_S0203_I00000481_P1437,improvised,dev,0,8,0,1,1,203,1437,3,V03,481,P0065_AA_ANCP_XXXX,You're going to play a game with your partner. It is intended to be silly and fun.\r\nYou are each given different lists of words or phrases below.,You're going to play a game with your partner. It is intended to be silly and fun.\r\nYou are each given different lists of words or phrases below.,ANCP,NaN,charades,Undisclosed,Undisclosed,Undisclosed,Undisclosed,Undisclosed,stranger,stranger
169,V03_S0203_I00000488_P1436,improvised,dev,0,17,0,1,1,203,1436,3,V03,488,P0046_DP_APCP_XXXX,"You will each see the same prompt:\r\nDiscuss the following statement with your partner: ""Surprises in life can be delightful","You will each see the same prompt:\r\nDiscuss the following statement with your partner: ""Surprises in life can be delightful or",APCP,NaN,ipc_conversation,3.75,4.25,3.1666666666666665,3.75,3.8333333333333335,stranger,stranger


In [221]:
# Merge filtered_si_df (file_id) with annotation_df (id)
annotation_partial_df = annotation_df.merge(
    filtered_si_df,
    left_on='id',
    right_on='file_id',
    how='left',
)

print("annotation_partial_df shape:", annotation_partial_df.shape)
annotation_partial_df.head()

annotation_partial_df shape: (293, 32)


,id,annotations:1P-IS,annotations:1P-R,annotations:3P-IS,annotations:3P-R,annotations:3P-V,file_id,label,split,batch_idx,archive_idx,has_imitator_movement,has_annotation_1p,has_annotation_3p,session_id,participant_id,vendor_id,vendor_Vid,prompt_hash,prompt_id_unique,participant_a_prompt_text,participant_b_prompt_text,ipc_a,ipc_b,interaction_type,extraversion_raw,agreeableness_raw,conscientiousness_raw,neuroticism_raw,openness_raw,relationship,relationship_detail
0,V03_S1035_I00000375_P3095,Deep in conversation.,topic interesting to me.,Participant is engaged and explaining with attention and focus.,Participant is emphasizing their explanation while maintaining focus and awareness of surroundings.,"Participant speaks and does hand gestures, looks around, and maintains a focused, normal facial expression.",V03_S1035_I00000375_P3095,improvised,train,65,3,0,1,1,1035,3095,3,V03,375,P0025_DP_APCN_XXXX,Tell your partner an opinion you have that you know is right. Explain your point of view to,"Your partner will share a point of view on something, discuss it with them.",APCN,NaN,ipc_conversation,2.4166666666666665,4.0,4.0,3.0833333333333335,4.25,stranger,stranger
1,V03_S1039_I00000373_P3206,"Searching for words, invested",Making sure I won't stumble on words while recalling names for future examples,Participant is in doubt,Participant is trying to recall names for the future,Participant moves their arm up and down multiple times gesturing size,V03_S1039_I00000373_P3206,improvised,train,65,44,0,1,1,1039,3206,3,V03,373,P0047_CC_ANCP_XXXX,You will each see the same prompt:,Your partner will tell you a story from their childhood. Listen to them and engage with the,ANCP,NaN,ipc_conversation,Undisclosed,Undisclosed,Undisclosed,Undisclosed,Undisclosed,stranger,stranger
2,V03_S1004_I00000541_P2024,"sure, maybe opinionated",speaking about film & art,The participant is demonstrative,Because the participant wanted to accentuate the role of the quality of the story,The participant curled the fingers and raised the hands to the chest's height like they were holding something,V03_S1004_I00000541_P2024,improvised,train,65,164,0,1,1,1004,2024,3,V03,541,P0001_DN_AMCM_AMCM,"Ask your partner about a topic for which you have an ""unpopular opinion"". Don't disclose how you feel about the topic, or what your opinion is. Simply ask your partner how they feel about it.","Your partner will ask you for your opinion on a topic. Explain your opinion briefly, and then try and get them to express their opinion. Press them to explain their position.",AMCM,AMCM,ipc_conversation,3.0833333333333335,4.916666666666667,3.0,2.25,5.0,stranger,stranger
3,V03_S1004_I00000372_P2024,"uplifted, happy, proud",to find a kindered heart,The participant is confident,The participant had a lot of knowledge in the topic and they were talking smoothly,The participant moved their hands non-stop,V03_S1004_I00000372_P2024,improvised,train,65,129,0,1,1,1004,2024,3,V03,372,P0040_MP_APCP_XXXX,Try to recall someone important from your childhood - could be a best friend or grandparent,Your partner will tell you a story from their childhood. Listen to them and engage with the,APCP,NaN,ipc_conversation,3.0833333333333335,4.916666666666667,3.0,2.25,5.0,stranger,stranger
4,V03_S1035_I00000371_P3095,Felt like i wanted to go back to the old days.,Due to picturing the Yankees and being a former baseball player.,"Participant is calm, focused, and patiently listening.",Participant is absorbing the information with calm attention and minimal distraction.,"Participant listens attentively with minimal body movement and slight finger movement, then wipes their face with their forearm.",V03_S1035_I00000371_P3095,improvised,train,65,54,0,1,1,1035,3095,3,V03,371,P0007_EO_ANCP_XXXX,"Ask your partner: ""Would you rather have the power to teleport or the ability to time travel?""",Your partner will pose a question to you. Have fun answering it.,ANCP,NaN,ipc_conversation,2.4166666666666665,4.0,4.0,3.0833333333333

In [222]:
# Save
annotation_partial_df.to_csv(ASSET_DIR / 'annotation_partial.csv', index=False)